# Employee Attrition Prediction – EDA and Modeling

This notebook walks through the full pipeline: loading data, EDA, preprocessing, model training with tuning, evaluation and prediction. It reuses the modules in `src/`, so the notebook and the scripts always agree.

**Run the cells top to bottom** (Kernel → Restart & Run All).

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")   # so we can import our own modules

import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)

from utils import DATA_PATH, MODEL_PATH, RANDOM_STATE, NUMERIC_FEATURES
from preprocessing import load_data, split_features_target, build_preprocessor
from train_model import get_candidate_models, evaluate, get_feature_importance
from predict import predict_attrition

sns.set_theme(style="whitegrid")

## 2. Load and inspect the data

In [ ]:
df = load_data()
print("Shape:", df.shape)
df.head()

In [ ]:
# Missing values per column (we handle these later with imputation)
df.isnull().sum()

In [ ]:
df.describe()

## 3. Exploratory Data Analysis

### 3.1 Class balance
Attrition datasets are skewed: far more employees stay than leave, so accuracy alone is misleading.

In [ ]:
df["Left"] = (df["Attrition"] == "Yes").astype(int)
print(f"Attrition rate: {df['Left'].mean():.1%}")
df["Attrition"].value_counts().plot(kind="bar", color=["#4c9", "#e66"], rot=0)
plt.title("Class distribution (Attrition)")
plt.show()

### 3.2 Attrition rate by department, overtime and job satisfaction

In [ ]:
for col in ["Department", "OverTime", "JobSatisfaction"]:
    print(f"\nAttrition rate (%) by {col}")
    print((df.groupby(col)["Left"].mean() * 100).round(1).sort_values(ascending=False))

### 3.3 Visualisation 1 – Attrition rate by department

In [ ]:
rate = df.groupby("Department")["Left"].mean().mul(100).sort_values(ascending=False)
plt.figure(figsize=(7, 4))
sns.barplot(x=rate.index, y=rate.values, hue=rate.index, palette="viridis", legend=False)
plt.ylabel("Attrition rate (%)")
plt.title("Attrition Rate by Department")
plt.show()

### 3.4 Visualisation 2 – Monthly income vs attrition

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="Attrition", y="MonthlyIncome", hue="Attrition", palette="Set2", legend=False)
plt.title("Monthly Income (Rs.) vs Attrition")
plt.show()

### 3.5 Visualisation 3 – Correlation heatmap

In [ ]:
numeric = df.select_dtypes("number").drop(columns=["EmployeeID"])
plt.figure(figsize=(9, 7))
sns.heatmap(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

print("Correlation with attrition (strongest first):")
numeric.corr()["Left"].drop("Left").sort_values(key=abs, ascending=False).round(3)

**EDA takeaways**

- Overtime employees leave far more often than others.
- Low job satisfaction is linked to much higher attrition.
- Higher income, longer tenure and higher age go with lower attrition.
- Sales and HR have the highest attrition rates.

## 4. Preprocessing and train/test split
We split **first**, then let the pipeline learn imputation/scaling/encoding from the training data only. This prevents data leakage.

In [ ]:
X, y = split_features_target(df.drop(columns=["Left"]))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Attrition rate train/test:", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))

## 5. Model training with hyperparameter tuning
Two models are compared. Class imbalance is handled with `class_weight="balanced"`. `GridSearchCV` (5-fold stratified) tunes each model on ROC-AUC.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fitted, rows = {}, {}

for name, (model, grid) in get_candidate_models().items():
    pipe = Pipeline([("prep", build_preprocessor()), ("model", model)])
    search = GridSearchCV(pipe, grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    fitted[name] = search.best_estimator_
    rows[name] = evaluate(search.best_estimator_, X_test, y_test)
    print(f"{name}: best params = {search.best_params_} | CV ROC-AUC = {search.best_score_:.3f}")

## 6. Model comparison

In [ ]:
results = pd.DataFrame(rows).T.round(3)
results

In [ ]:
results.plot(kind="bar", figsize=(8, 4.5), colormap="Set2")
plt.title("Model Comparison"); plt.ylim(0, 1); plt.xticks(rotation=0)
plt.legend(loc="lower right", ncol=2)
plt.show()

### Best model
We pick the model with the highest ROC-AUC (F1-score breaks ties). ROC-AUC measures how well the model ranks leavers above stayers and does not depend on the 0.5 threshold, which suits imbalanced data. Recall is also important here: missing an employee who is about to leave costs more than a false alarm.

In [ ]:
best_name = results.sort_values(["ROC-AUC", "F1-score"], ascending=False).index[0]
best_model = fitted[best_name]
print("Best model:", best_name)

## 7. Detailed evaluation of the best model

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Stayed", "Left"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Stayed", "Left"], ax=axes[0], cmap="Blues")
axes[0].set_title("Confusion Matrix")
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
axes[1].set_title("ROC Curve")
plt.tight_layout(); plt.show()

## 8. Visualisation 4 – Feature importance

In [ ]:
importance = get_feature_importance(best_model)
top = importance.head(10)[::-1]
plt.figure(figsize=(8, 5))
plt.barh(top.index, top.values, color="teal")
plt.title(f"Top Factors Influencing Attrition ({best_name})")
plt.xlabel("Importance")
plt.show()
importance.head(5).round(3)

*Note:* `OverTime_Yes` and `OverTime_No` both appear because One-Hot Encoding creates one column per category. They describe the same factor (overtime).

## 9. Save the model and test a prediction

In [ ]:
joblib.dump({"pipeline": best_model, "model_name": best_name}, MODEL_PATH)
print("Saved to", MODEL_PATH)

In [ ]:
sample = {"Age": 28, "Department": "Sales", "MonthlyIncome": 30000, "YearsAtCompany": 1,
          "JobSatisfaction": 1, "WorkLifeBalance": 2, "PerformanceRating": 3,
          "OverTime": "Yes", "DistanceFromHome": 30}
predict_attrition(sample)

## 10. Conclusion

- The pipeline handles missing values, encoding, scaling and class imbalance without data leakage.
- Logistic Regression gave the best ROC-AUC and recall, and is easy to explain.
- Overtime, job satisfaction, income and work-life balance are the main drivers of attrition.
- The saved model is used by `src/predict.py` and the Streamlit app (`app.py`).